# 模块 Z · 检查 config.yaml 和样本表（每套流程的第一步）   [Python]

- 作用：在跑几个小时之前，先确认配置字段齐全、取值合法、和所选流程一致，样本路径都存在。
- 依据：schemas/config.schema.json

**对应的流程脚本**：`scripts/Z_provenance/validate_config.py`（代码相同，流程运行的是 .py）

**用法**：先把下面第一个代码格里的 `project_dir` 改成项目目录，然后从上到下逐格运行。

In [ ]:
# 步骤：切换到项目目录（里面有 config.yaml、samples.tsv 和 results/）
# 输入：project_dir（改成自己的项目目录）
# 输出：无（之后的代码都以项目目录为当前目录）
import os
project_dir = "."      # 例如 "/data/my_visium_project"
os.chdir(project_dir)
os.getcwd()

## 1. 导入包

In [ ]:
# 步骤：导入包
# 输入：无
# 输出：无
from pathlib import Path
import os
import json
import jsonschema
import pandas as pd
import yaml

## 2. 读取配置和 schema

In [ ]:
# 步骤：读取配置和 schema
# 输入：config.yaml；skill_dir（本 skill 的安装位置）下的 schemas/config.schema.json
# 输出：cfg、schema
skill_dir = Path(os.environ.get("SPATIAL_SKILL_DIR", "~/.claude/skills/spatial-downstream")).expanduser()   # 流程脚本会自动设好；单独运行且装在 Codex 里时改成 ~/.codex/skills/spatial-downstream
cfg = yaml.safe_load(open("config.yaml"))
schema = json.load(open(skill_dir / "schemas" / "config.schema.json"))

这一步得到：cfg、schema

## 3. 按 schema 检查配置；不合法时直接报错并指出字段

In [ ]:
# 步骤：按 schema 检查配置；不合法时直接报错并指出字段
# 输入：cfg、schema
# 输出：无（不合法时抛出 ValidationError）
jsonschema.validate(cfg, schema)

这一步得到：无（不合法时抛出 ValidationError）

## 4. 检查样本表的列名和每个样本路径是否存在

In [ ]:
# 步骤：检查样本表的列名和每个样本路径是否存在
# 输入：samples.tsv
# 输出：samples；打印每个路径是否存在
samples = pd.read_csv(cfg["samples"], sep="\t")
assert list(samples.columns[:3]) == ["sample", "condition", "path"], "samples.tsv 前三列必须是 sample / condition / path"
samples["exists"] = samples["path"].map(lambda p: Path(p).exists())
print(samples)
assert samples["exists"].all(), "有样本路径不存在，见上表"

这一步得到：samples；打印每个路径是否存在